# 01 · EDSR ×3 — Sentinel-2(10 m) → IKONOS(3.3333 m) 초해상화

Colab 실습용 소형 번들. 학습 40패치 / 검증 10패치 / 테스트 1씬으로 **전체 파이프라인을
10분 안에** 한 바퀴 돌린다.

| | 내용 |
|---|---|
| 학습 입력 | 합성 LR (`g_LR`, HR에서 만든 것) 128px → HR 384px, 6개 도시 40쌍 |
| 검증 입력 | **실제 Sentinel-2 LR** 128px, 학습에 안 쓴 홀드아웃 10씬에서 1장씩 |
| 테스트 | Incheon 씬 GeoTIFF 1장 (2001×2001, GT 없음) |

학습은 합성 LR로, 검증은 실제 S2로 한다. 이 비대칭이 이 프로젝트의 핵심이다 —
검증 PSNR에는 **합성↔실제 도메인 갭**이 통째로 들어 있다.

> **먼저 GPU를 켤 것**: 런타임 → 런타임 유형 변경 → 하드웨어 가속기 = **T4 GPU**

## 0. 런타임 확인

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  ', torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU가 꺼져 있습니다. 런타임 → 런타임 유형 변경 → T4 GPU')

## 1. 번들 가져오기

`colab/` 폴더(42 MB)를 Colab VM으로 가져온다. 세 가지 경로 중 하나를 고르면 된다.

| `SOURCE` | 방법 | 언제 쓰나 |
|---|---|---|
| `'github'` | 공개 저장소를 `git clone` | **가장 간단**. 링크만 있으면 누구나 어디서나 실행 |
| `'drive'` | 내 Drive의 zip을 복사 | 데이터를 공개하고 싶지 않을 때 |
| `'url'` | 직접 다운로드 링크에서 받기 | 사내 스토리지·S3 등 |

아래 셀 맨 위 세 줄만 자기 환경에 맞게 고친다.

In [ ]:
import os, shutil, subprocess, glob

SOURCE      = 'github'                                        # github | drive | url
GITHUB_REPO = 'https://github.com/BWMIN-Hub/SR_practice.git'  # SOURCE='github' 일 때
DRIVE_ZIP   = '/content/drive/MyDrive/sr_colab/colab.zip'     # SOURCE='drive' 일 때
BUNDLE_URL  = ''                                              # SOURCE='url' 일 때


def _mount_drive():
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')


def _find_root(base):
    """dataset/ models/ notebooks/ 를 모두 가진 폴더를 찾는다.

    저장소 루트가 colab/ 자체든, colab/ 을 품은 상위 폴더든 양쪽 다 찾아낸다.
    """
    for d, subs, _ in os.walk(base):
        if {'dataset', 'models', 'notebooks'} <= set(subs):
            return d
    return None


def fetch():
    """번들을 /content 아래로 가져오고 그 루트 경로를 돌려준다."""
    hit = _find_root('/content')
    if hit:                                      # 이미 있으면 다시 받지 않는다
        return hit

    if SOURCE == 'github':
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, '/content/repo'],
                       check=True)
    elif SOURCE in ('drive', 'url'):
        zp = '/content/colab.zip'
        if SOURCE == 'drive':
            _mount_drive()
            assert os.path.exists(DRIVE_ZIP), f'Drive에 없습니다: {DRIVE_ZIP}'
            shutil.copy(DRIVE_ZIP, zp)           # 순차 읽기 1회
        else:
            subprocess.run(['wget', '-q', '-O', zp, BUNDLE_URL], check=True)
        subprocess.run(['unzip', '-q', '-o', zp, '-d', '/content/repo'], check=True)
    else:
        raise ValueError(SOURCE)

    hit = _find_root('/content/repo')
    assert hit, '받은 내용에서 dataset/·models/·notebooks/ 를 가진 폴더를 찾지 못했습니다'
    return hit


ROOT  = fetch()
DATA  = f'{ROOT}/dataset'
MODEL = f'{ROOT}/models/01_edsr_x3'
CODE  = f'{MODEL}/code'
assert os.path.isdir(CODE), CODE
print('번들 준비 완료:', ROOT)

### 체크포인트를 Drive로 빼둔다

Colab 세션은 **최대 12시간, 유휴 90분이면 끊기고** `/content`는 전부 사라진다.
학습 산출물이 나오는 `code/experiment/`만 Drive로 심볼릭 링크해두면 세션이 끊겨도 남는다
(체크포인트는 6 MB짜리라 부담 없다).

In [ ]:
SAVE_TO_DRIVE = True   # Drive를 안 쓸 거면 False

if SAVE_TO_DRIVE:
    _mount_drive()
    dst = '/content/drive/MyDrive/sr_colab/experiment'
    os.makedirs(dst, exist_ok=True)
    if not os.path.islink(f'{CODE}/experiment'):
        shutil.rmtree(f'{CODE}/experiment', ignore_errors=True)
        os.symlink(dst, f'{CODE}/experiment')
    print('experiment ->', os.path.realpath(f'{CODE}/experiment'))
else:
    os.makedirs(f'{CODE}/experiment', exist_ok=True)

### 의존성

torch·opencv·scikit-image·imageio·matplotlib은 Colab에 이미 깔려 있다.
GeoTIFF를 읽는 **rasterio만** 추가로 설치한다.

In [ ]:
!pip install -q rasterio
!apt-get install -qq -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm

# Colab 기본 이미지에는 한글 폰트가 없어 그래프 제목이 네모로 깨진다
_font = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    mpl.rc('font', family='NanumGothic')
mpl.rc('axes', unicode_minus=False)

import rasterio
print('rasterio', rasterio.__version__, '| 한글폰트', os.path.exists(_font))

## 2. 데이터 둘러보기

먼저 데이터가 어떻게 생겼는지 본다. `HR = LR × 3`이 정확히 성립하고 지리 범위가 같아서
리샘플링이 필요 없다는 것이 이 데이터셋의 전제다.

In [ ]:
from collections import Counter

for split in ['training', 'validation']:
    hr = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))
    lr = sorted(glob.glob(f'{DATA}/{split}/LR_bicubic/X3/*.png'))
    city = Counter(os.path.basename(f).rsplit('_y', 1)[0].rsplit('_', 1)[0] for f in hr)
    print(f'{split:11s} HR {len(hr):3d}장 / LR {len(lr):3d}장   {dict(city)}')

print('\ntest      ', [os.path.basename(f) for f in glob.glob(f'{DATA}/test/*.tif')])

In [ ]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt

def preview(split, n=4):
    hr_files = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))[:n]
    fig, ax = plt.subplots(2, len(hr_files), figsize=(3.2 * len(hr_files), 6.8))
    for i, f in enumerate(hr_files):
        stem = os.path.basename(f)[:-4]
        hr = imageio.imread(f)
        lr = imageio.imread(f'{DATA}/{split}/LR_bicubic/X3/{stem}x3.png')
        ax[0, i].imshow(lr); ax[0, i].set_title(f'LR {lr.shape[1]}x{lr.shape[0]}', fontsize=9)
        ax[1, i].imshow(hr); ax[1, i].set_title(f'HR {hr.shape[1]}x{hr.shape[0]}', fontsize=9)
        ax[1, i].set_xlabel(stem.rsplit('_y', 1)[0], fontsize=7)
        for a in (ax[0, i], ax[1, i]): a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'{split}  (위: 입력 LR 10m / 아래: 정답 HR 3.33m)')
    plt.tight_layout(); plt.show()

preview('training')
preview('validation')

위쪽 LR을 아래쪽 HR만큼 선명하게 만드는 것이 과제다.

**training과 validation을 비교해서 보라.** training의 LR은 HR을 3배 축소해 만든 **합성**
영상이라 HR과 색·밝기가 정확히 일치한다. validation의 LR은 **실제로 촬영된 Sentinel-2**라
촬영 시기·센서가 달라 색이 어긋나 있다. 이 차이가 뒤에서 검증 PSNR을 눌러앉히는 원인이다.

## 3. 학습 (fine-tune)

밑바닥부터 학습하면 Colab에서 몇 시간이 걸리므로, 같은 데이터로 30 epoch 학습해둔
체크포인트에서 이어서 돌린다.

`run_train.sh`는 전부 환경변수로 조절한다:

| 변수 | 뜻 |
|---|---|
| `DATA=COLAB` | `src/data/colab.py`의 `COLAB` 클래스를 쓴다 |
| `DIR_DATA` | 데이터셋 루트 (`training/`·`validation/`의 부모) |
| `EPOCHS` | **실제 학습은 `EPOCHS-1`회**. EDSR이 `epoch >= epochs`에서 멈춘다 |
| `TEST_EVERY` | epoch당 iteration. epoch 샘플수 = `batch(16) × TEST_EVERY` |
| `RESET=0` | `experiment/`를 지우지 않고 마지막 체크포인트에서 재개 (세션 끊김 대비) |

In [ ]:
import time

env = dict(os.environ,
    GPU='0',
    DATA='COLAB',
    DIR_DATA=DATA,
    EPOCHS='11',            # 실제 10 epoch (EDSR off-by-one)
    DECAY='5-8',
    LR='1e-4',
    TEST_EVERY='100',       # 40패치짜리 소형셋 -> epoch당 1600샘플
    PRINT_EVERY='20',       # TEST_EVERY 보다 작아야 loss 로그가 남는다
    N_THREADS='2',          # Colab은 vCPU 2개
    SAVE='edsr_colab_x3',
    SAVE_RESULTS='0',
    RESET='1',              # 이어서 학습할 때는 '0'
    PRETRAIN=f'{MODEL}/checkpoints/edsr_ikonos_x3_best.pt',
)

t0 = time.time()
p = subprocess.run(['bash', f'{CODE}/run_train.sh'], env=env,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout.splitlines():
    if 'Making a binary' in line or 'it/s]' in line: continue
    print(line)
print(f'\n소요 {time.time() - t0:.0f}초')

## 4. 학습 곡선

`experiment/edsr_colab_x3/log.txt`에 epoch별 L1 loss와 검증 PSNR이 쌓인다.

In [ ]:
import numpy as np
import torch

EXP = f'{CODE}/experiment/edsr_colab_x3'
# EDSR 이 epoch 별 평균을 텐서로 저장해준다 (log.txt 를 정규식으로 긁는 것보다 안전)
loss = torch.load(f'{EXP}/loss_log.pt').flatten().numpy()
psnr = torch.load(f'{EXP}/psnr_log.pt').flatten().numpy()
ep = np.arange(1, len(loss) + 1)
best = int(np.argmax(psnr))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep, loss, 'o-')
ax[0].set_title('학습 L1 loss — 합성 LR 도메인')
ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
ax[1].plot(ep, psnr, 'o-')
ax[1].plot(best + 1, psnr[best], 'r*', ms=15, label=f'best {psnr[best]:.3f} @ep{best+1}')
ax[1].set_title('검증 PSNR — 실제 S2 LR, 홀드아웃 10패치')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'학습 loss  {loss[0]:.2f} -> {loss[-1]:.2f}  (계속 내려간다)')
print(f'검증 PSNR  best {psnr[best]:.3f} dB @ epoch {best+1} / 마지막 {psnr[-1]:.3f} dB')

**학습 loss는 계속 내려가는데 검증 PSNR은 초반에 best를 찍고 정체할 것이다.**
버그가 아니라 이 프로젝트의 핵심 관찰이다. 모델은 학습 도메인(합성 LR)에서 계속
좋아지지만, 실제 S2 입력에는 그만큼 따라오지 못한다. 남은 차이는 기하 문제가 아니라
센서 특성·촬영 시기·방사 차이다.

## 5. 추론 — 실제 씬 전체

`infer.py`는 LR 기준 256px 타일에 16px 겹침으로 잘라 이어붙이고, 입력의 CRS·지리 범위를
그대로 둔 채 픽셀 크기만 1/3로 줄여 GeoTIFF로 쓴다. 결과는 원본과 **정확히 같은 영역**을
덮으므로 QGIS에서 바로 겹쳐볼 수 있다.

> `--chop`은 절대 쓰지 말 것. 업스트림 `forward_chop`에 배치 차원이 깨지는 버그가 있다.

In [ ]:
OUTDIR = '/content/results'
weight = f'{CODE}/experiment/edsr_colab_x3/model/model_best.pt'

p = subprocess.run(['python', 'infer.py', '--weight', weight,
                    '--input', f'{DATA}/test', '--output', OUTDIR],
                   cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(p.stdout[-1500:])

In [ ]:
# 좌표가 보존됐는지 확인 — 실습에서 가장 자주 틀리는 부분이다
src_tif = glob.glob(f'{DATA}/test/*.tif')[0]
out_tif = glob.glob(f'{OUTDIR}/*_SRx3.tif')[0]

with rasterio.open(src_tif) as a, rasterio.open(out_tif) as b:
    print(f'입력  {a.width}x{a.height}  {a.res[0]:.4f} m  {a.crs}')
    print(f'출력  {b.width}x{b.height}  {b.res[0]:.4f} m  {b.crs}')
    same = np.allclose(np.array(a.bounds), np.array(b.bounds), atol=1e-6)
    print(f'\n지리 범위 일치: {same}')
    assert same, '좌표가 어긋났습니다'

## 6. Bicubic과 비교

단순 보간(bicubic)보다 나은지 본다. GT가 없으므로 PSNR은 못 내고,
**라플라시안 표준편차**(= 고주파 양)와 육안 비교로 판단한다.

In [ ]:
import cv2

with rasterio.open(src_tif) as s:
    lr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
with rasterio.open(out_tif) as s:
    sr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
bic = cv2.resize(lr, (lr.shape[1] * 3, lr.shape[0] * 3), interpolation=cv2.INTER_CUBIC)

def lap_std(img):
    return float(cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'Bicubic x3   lap_std {lap_std(bic):6.2f}   평균밝기 {bic.reshape(-1,3).mean(0).round(1)}')
print(f'EDSR    x3   lap_std {lap_std(sr):6.2f}   평균밝기 {sr.reshape(-1,3).mean(0).round(1)}')
print('\n※ lap_std는 같은 격자끼리만 비교할 것. 10m 원본 LR의 값은 픽셀 계단 때문에 크게 나온다.')

In [ ]:
# 텍스처가 많은 구역을 골라 확대 비교
gray = cv2.cvtColor(sr, cv2.COLOR_RGB2GRAY)
S = 300
best, bs = (0, 0), -1
for y in range(0, sr.shape[0] - S, S):
    for x in range(0, sr.shape[1] - S, S):
        v = gray[y:y+S, x:x+S].std()
        if v > bs: best, bs = (y, x), v
y, x = best

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, img, t in zip(ax, [cv2.resize(lr[y//3:y//3+S//3, x//3:x//3+S//3], (S, S),
                                     interpolation=cv2.INTER_NEAREST), bic[y:y+S, x:x+S], sr[y:y+S, x:x+S]],
                     ['입력 LR (10 m, 최근접확대)', 'Bicubic ×3', 'EDSR ×3']):
    a.imshow(img); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 7. 결과 저장

`/content`는 세션이 끝나면 사라진다. 남길 것은 Drive로 옮긴다.
(`experiment/`는 1번 셀에서 이미 Drive로 링크해뒀다.)

In [ ]:
if SAVE_TO_DRIVE:
    dst = '/content/drive/MyDrive/sr_colab/results'
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{OUTDIR}/*'):
        shutil.copy(f, dst)
    print('저장:', os.listdir(dst))

---

## 알아둘 함정

1. **`EPOCHS=N`은 실제로 N−1회 학습한다.** `Trainer.terminate()`가 `epoch >= args.epochs`에서
   멈춘다. 10 epoch를 원하면 `EPOCHS=11`. `EPOCHS=1`은 아무것도 학습하지 않는다.
2. **`--chop`은 쓰지 말 것.** 업스트림 `forward_chop`의 `zip(*x_chops)`가 4D 텐서를 3D로
   떨어뜨려 터진다. 큰 영상은 `infer.py`의 타일링을 쓴다.
3. **데이터를 바꿔 넣으면 `dataset/bin/`을 지울 것.** PNG를 `.pt`로 캐싱하는데 파일명이 같으면
   옛 캐시를 그대로 재사용한다.
4. **Drive에서 직접 학습하지 말 것.** 작은 파일 반복 읽기가 극단적으로 느리다.
   zip을 `/content`로 복사해 풀고 거기서 학습한다.
5. **세션이 끊기면 `/content`는 전부 사라진다.** `experiment/`는 Drive 링크로,
   재개는 `RESET='0'`으로 한다.
6. **검증 PSNR이 낮다고 학습이 실패한 게 아니다.** 검증 입력은 실제 Sentinel-2라
   합성 LR로 학습한 모델에게는 다른 도메인이다.